# Feeana: Dual-Head Fine-Tuning + ONNX Export (Google Colab T4)

Minimal, reproducible notebook targeting Google Colab T4 GPU runtime.

### Overview:
- **Base Model**: configurable — set once in the *Config* cell below (`bert-base-multilingual-cased` for mBERT, or the DistilXLM-R default). Public models, no HF token required.
- **Architecture**: Shared encoder + 15-way `issue` head + 3-way `polarity` head
- **Training**: Full fine-tuning of the shared encoder
- **Export**: ONNX FP32 → INT8 quantization + smoke test (Steps 9–10)
- **Dependencies**: Minimal installation targeting training dependencies without touching preinstalled Colab numpy/pandas

### Config: Select Base Model

**This is the ONE line you change** to switch which model is fine-tuned and exported.

- `bert-base-multilingual-cased` → mBERT (output tag: `mbert`)
- `nreimers/mMiniLMv2-L12-H384-distilled-from-XLMR-Large` → DistilXLM-R (default, tag: `distilxlmr`)

All downstream steps (training, diagnostics, ONNX export, smoke test) read this env var automatically.

In [ ]:
import os
os.environ["FEEANA_MODEL_NAME"] = ""
print(f"[CONFIG] Base model: {os.environ['FEEANA_MODEL_NAME']}")

### Step 1: Verify CUDA & GPU Environment

In [ ]:
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device Name: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: GPU is not enabled! Go to Runtime > Change runtime type > Select T4 GPU.")

!nvidia-smi

### Step 2: Install Minimal Required Dependencies

Upgrades `torchao` to `>=0.16.0` while avoiding unnecessary `-U` upgrades or version locks on preinstalled Colab packages (`numpy`, `pandas`, `torch`).

In [ ]:
!pip install -q \
  "torchao>=0.16.0" \
  "transformers>=4.38.0" \
  "datasets>=2.18.0" \
  "evaluate>=0.4.0" \
  "accelerate>=0.27.0" \
  "scikit-learn>=1.3.0" \
  "onnx>=1.15.0" \
  "onnxruntime>=1.17.0"

### Step 3: Verify Core Package Imports & Versions

In [ ]:
import sys
import subprocess

cmd = [
    sys.executable, "-c",
    "import torch, torchao, transformers; " \
    "print('torch:       ', torch.__version__); " \
    "print('torchao:     ', torchao.__version__); " \
    "print('transformers:', transformers.__version__)"
]
res = subprocess.run(cmd, capture_output=True, text=True, check=True)
print(res.stdout)
print("[PASS] Core imports verified without version conflicts!")

### Step 4: Smoke Test — Dual-Head Model Instantiation & Forward Pass (with AMP/FP16)

Verifies that `DualHeadModel` instantiates and executes a forward pass under AMP without FP16/float32 dtype mismatches.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, "scripts/training")

import torch
from finetune import DualHeadModel, MODEL_NAME, NUM_ISSUES, NUM_POLARITIES

print(f"[SMOKE TEST] Loading base model '{MODEL_NAME}'...")
model = DualHeadModel(MODEL_NAME, NUM_ISSUES, NUM_POLARITIES)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

dummy_ids = torch.zeros((2, 16), dtype=torch.long, device=device)
dummy_mask = torch.ones((2, 16), dtype=torch.long, device=device)

use_amp = (device.type == "cuda")
with torch.no_grad():
    with torch.amp.autocast(device_type=device.type, enabled=use_amp):
        out = model(dummy_ids, dummy_mask)

assert out["issue_logits"].shape == (2, NUM_ISSUES), f"Issue logits shape mismatch: {out['issue_logits'].shape}"
assert out["polarity_logits"].shape == (2, NUM_POLARITIES), f"Polarity logits shape mismatch: {out['polarity_logits'].shape}"

print(f"\n[PASS] DualHeadModel successfully initialized and verified under AMP on {device}!")
print(f"  - Issue logits shape:    {out['issue_logits'].shape}")
print(f"  - Polarity logits shape: {out['polarity_logits'].shape}")

### Step 5: Verify Dataset & Script Files

**Upload the entire `scripts/training/` folder** (all `.py` scripts plus the `data/` subfolder), not just the files checked below. `finetune.py`, `checkpoint_paths.py`, `export_model_onnx.py`, and `smoke_test_onnx.py` import each other and are all required by Steps 6–10.

In [ ]:
from pathlib import Path
import pandas as pd

required_files = [
    Path("scripts/training/finetune.py"),
    Path("scripts/training/checkpoint_paths.py"),
    Path("scripts/training/export_model_onnx.py"),
    Path("scripts/training/smoke_test_onnx.py"),
    Path("scripts/training/data/train.csv"),
    Path("scripts/training/data/val.csv"),
    Path("scripts/training/data/test.csv"),
]

missing = []
for f in required_files:
    if f.exists():
        if f.suffix == ".csv":
            df = pd.read_csv(f)
            print(f"[FOUND] {f} ({len(df):,} rows)")
        else:
            print(f"[FOUND] {f}")
    else:
        missing.append(str(f))

if missing:
    raise FileNotFoundError(f"Missing required files: {missing}. Please upload the scripts/ folder to Colab.")
else:
    print("\n[READY] All required training files and data splits verified!")

### Step 6: Execute Phase 2 Fine-Tuning Run

The base model comes from the *Config* cell (`FEEANA_MODEL_NAME`), so no `--model-name` flag is needed here. Batch 16 fits the T4 for the DistilXLM-R base; if you hit a CUDA OOM with mBERT (~110M params), do NOT lower `--batch-size` to 8.

In [ ]:
!python scripts/training/finetune.py --epochs 5 --batch-size 16 --lr 2e-5

### Step 7: Verify Checkpoint Artifacts & Saved Model

In [ ]:
import sys
from pathlib import Path
import json

sys.path.insert(0, "scripts/training")
from finetune import MODEL_NAME, resolve_tag

tag = resolve_tag(MODEL_NAME)
ckpt_file = Path("scripts/training/checkpoints") / tag / "best_model.pt"
json_file = Path("scripts/training/checkpoints") / tag / "label_mappings.json"

if ckpt_file.exists() and json_file.exists():
    size_mb = ckpt_file.stat().st_size / (1024 * 1024)
    print(f"[SUCCESS] Checkpoint saved: {ckpt_file} ({size_mb:.2f} MB)")
    with open(json_file, encoding="utf-8") as f:
        mappings = json.load(f)
    print(f"[SUCCESS] Label mappings saved: {json_file}")
    print(f"  - Issue labels ({mappings['issue']['num_labels']}): {list(mappings['issue']['id2label'].values())[:3]}...")
    print(f"  - Polarity labels ({mappings['polarity']['num_labels']}): {list(mappings['polarity']['id2label'].values())}")
    print("\nTraining completed successfully! Download scripts/training/checkpoints/ for Phase 3/4.")
else:
    print("WARNING: Checkpoints not found. Please review training log above.")

### Step 8: Post-Training Validation Diagnostics

Runs comprehensive diagnostics on the trained model's validation predictions:
- Per-class prediction distribution vs actual labels
- 15x15 confusion matrix
- Per-class precision, recall, F1
- Overall accuracy and Macro-F1
- Sample predictions with confidence scores
- Sanity checks for label/target integrity

In [ ]:
import sys, json
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from pathlib import Path
from collections import Counter
from torch.utils.data import DataLoader
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score
)

sys.path.insert(0, 'scripts/training')
from finetune import (
    DualHeadModel, FeedbackDataset, compute_weights,
    ISSUE_LABELS, ISSUE_LABEL2ID, ISSUE_ID2LABEL,
    POLARITY_LABELS, POLARITY_LABEL2ID, POLARITY_ID2LABEL,
    MODEL_NAME, NUM_ISSUES, NUM_POLARITIES, MAX_LEN,
    resolve_tag,
)
from transformers import AutoTokenizer
import torch.nn as nn

CKPT = Path('scripts/training/checkpoints') / resolve_tag(MODEL_NAME) / 'best_model.pt'
VAL_CSV = Path('scripts/training/data/val.csv')
TRAIN_CSV = Path('scripts/training/data/train.csv')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

print('\n' + '='*80)
print('  SANITY CHECKS')
print('='*80)

train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)

csv_issues = sorted(train_df['issue'].unique())
assert csv_issues == ISSUE_LABELS, f'MISMATCH: CSV issues {csv_issues} vs ISSUE_LABELS {ISSUE_LABELS}'
print('[PASS] ISSUE_LABELS matches CSV issue column values exactly.')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
val_ds = FeedbackDataset(VAL_CSV, tokenizer)

print('\nSample target verification (first 5):')
for i in range(min(5, len(val_ds))):
    item = val_ds[i]
    row = val_df.iloc[i]
    exp_issue_id = ISSUE_LABEL2ID[row['issue']]
    exp_pol_id = POLARITY_LABEL2ID[row['polarity']]
    actual_issue_id = item['issue_label'].item()
    actual_pol_id = item['polarity_label'].item()
    ok = 'OK' if (actual_issue_id == exp_issue_id and actual_pol_id == exp_pol_id) else 'FAIL'
    print(f'  [{i}] issue: csv={row["issue"]} -> exp_id={exp_issue_id}, got={actual_issue_id} | '
          f'polarity: csv={row["polarity"]} -> exp_id={exp_pol_id}, got={actual_pol_id} [{ok}]')
print('[PASS] Issue and polarity targets mapped independently and correctly.')

train_ds = FeedbackDataset(TRAIN_CSV, tokenizer)
issue_weights = compute_weights(train_ds.issue_ids, NUM_ISSUES, device)
polarity_weights = compute_weights(train_ds.polarity_ids, NUM_POLARITIES, device)
print(f'\nIssue class weights (15): {[round(w, 4) for w in issue_weights.cpu().tolist()]}')
print(f'Polarity class weights (3): {[round(w, 4) for w in polarity_weights.cpu().tolist()]}')

issue_crit = nn.CrossEntropyLoss(weight=issue_weights)
polarity_crit = nn.CrossEntropyLoss(weight=polarity_weights)

model = DualHeadModel(MODEL_NAME, NUM_ISSUES, NUM_POLARITIES)

if CKPT.exists():
    ckpt = torch.load(CKPT, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    print(f'\n[INFO] Loaded checkpoint: epoch={ckpt.get("epoch")}, '
          f'val_issue_F1={ckpt.get("val_issue_macro_f1")}, '
          f'val_pol_F1={ckpt.get("val_polarity_macro_f1")}')
else:
    print('[WARN] No checkpoint found, using randomly initialized model!')

model.to(device)
model.eval()

dummy_ids = torch.zeros((2, 16), dtype=torch.long, device=device)
dummy_mask = torch.ones((2, 16), dtype=torch.long, device=device)
with torch.no_grad():
    out = model(dummy_ids, dummy_mask)
    l_i = issue_crit(out['issue_logits'], torch.tensor([0, 14], device=device))
    l_p = polarity_crit(out['polarity_logits'], torch.tensor([0, 1], device=device))
print(f'\nDummy loss: issue={l_i.item():.4f}, polarity={l_p.item():.4f}, total={l_i.item()+l_p.item():.4f}')
assert l_i.item() > 0 and not torch.isnan(l_i), 'Issue loss is zero or NaN!'
print('[PASS] Issue loss is non-zero and included in total loss.')

print('\n' + '='*80)
print('  FULL VALIDATION INFERENCE')
print('='*80)

val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)

all_issue_preds = []
all_issue_targets = []
all_issue_confs = []
all_pol_preds = []
all_pol_targets = []
all_issue_loss_vals = []
all_pol_loss_vals = []
sample_records = []

with torch.no_grad():
    for b_idx, batch in enumerate(val_loader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        issue_labels = batch['issue_label'].to(device)
        polarity_labels = batch['polarity_label'].to(device)

        out = model(input_ids, attention_mask)
        issue_logits = out['issue_logits']
        pol_logits = out['polarity_logits']

        l_i = issue_crit(issue_logits, issue_labels)
        l_p = polarity_crit(pol_logits, polarity_labels)
        all_issue_loss_vals.append(l_i.item())
        all_pol_loss_vals.append(l_p.item())

        issue_probs = F.softmax(issue_logits, dim=-1)
        issue_preds = issue_logits.argmax(dim=-1)
        pol_preds = pol_logits.argmax(dim=-1)
        confs = issue_probs.max(dim=-1).values

        all_issue_preds.extend(issue_preds.cpu().tolist())
        all_issue_targets.extend(issue_labels.cpu().tolist())
        all_issue_confs.extend(confs.cpu().tolist())
        all_pol_preds.extend(pol_preds.cpu().tolist())
        all_pol_targets.extend(polarity_labels.cpu().tolist())

        start = b_idx * 32
        for i in range(len(issue_labels)):
            if len(sample_records) < 30:
                sample_records.append({
                    'text': val_ds.texts[start + i],
                    'actual_issue': ISSUE_ID2LABEL[issue_labels[i].item()],
                    'pred_issue': ISSUE_ID2LABEL[issue_preds[i].item()],
                    'confidence': issue_probs[i, issue_preds[i]].item(),
                    'actual_polarity': POLARITY_ID2LABEL[polarity_labels[i].item()],
                    'pred_polarity': POLARITY_ID2LABEL[pol_preds[i].item()],
                })

total_val = len(all_issue_targets)
overall_acc = accuracy_score(all_issue_targets, all_issue_preds)
macro_f1 = f1_score(all_issue_targets, all_issue_preds, average='macro', zero_division=0)
pol_macro_f1 = f1_score(all_pol_targets, all_pol_preds, average='macro', zero_division=0)
avg_issue_loss = np.mean(all_issue_loss_vals)
avg_pol_loss = np.mean(all_pol_loss_vals)

print(f'\nTotal validation samples: {total_val}')
print(f'Average issue loss:      {avg_issue_loss:.4f}')
print(f'Average polarity loss:   {avg_pol_loss:.4f}')
print(f'Issue overall accuracy:  {overall_acc:.4f} ({overall_acc*100:.2f}%)')
print(f'Issue Macro-F1:          {macro_f1:.4f}')
print(f'Polarity Macro-F1:       {pol_macro_f1:.4f}')

print('\n' + '='*80)
print('  ISSUE CLASS DISTRIBUTION (ACTUAL vs PREDICTED)')
print('='*80)
pred_counts = Counter(all_issue_preds)
true_counts = Counter(all_issue_targets)

print(f'{"Idx":<5} {"Class Name":<30} {"Actual":>8} {"Predicted":>10} {"Pred %":>8}')
print('-' * 65)
for idx, name in enumerate(ISSUE_LABELS):
    a = true_counts.get(idx, 0)
    p = pred_counts.get(idx, 0)
    pct = (p / total_val) * 100
    flag = ' <<' if p == 0 else ''
    print(f'[{idx:2d}] {name:<30} {a:>8d} {p:>10d} {pct:>7.2f}%{flag}')

unique_predicted = len(pred_counts)
print(f'\nUnique issue classes predicted: {unique_predicted} / {NUM_ISSUES}')
if unique_predicted <= 3:
    print('[ALERT] Possible class collapse detected!')

print('\n' + '='*80)
print('  PER-CLASS PRECISION / RECALL / F1')
print('='*80)
report = classification_report(
    all_issue_targets, all_issue_preds,
    labels=list(range(NUM_ISSUES)),
    target_names=ISSUE_LABELS,
    digits=4, zero_division=0
)
print(report)

print('\n' + '='*80)
print('  15x15 ISSUE CONFUSION MATRIX')
print('  Rows = actual, Columns = predicted')
print('='*80)
cm = confusion_matrix(all_issue_targets, all_issue_preds, labels=list(range(NUM_ISSUES)))

hdr = '     ' + ' '.join([f'{i:4d}' for i in range(NUM_ISSUES)])
print(hdr)
print('    ' + '-' * (NUM_ISSUES * 5 + 1))
for i in range(NUM_ISSUES):
    row_str = ' '.join([f'{cm[i,j]:4d}' for j in range(NUM_ISSUES)])
    print(f'{i:2d} | {row_str}')

print('\nConfusion matrix legend:')
for i, name in enumerate(ISSUE_LABELS):
    print(f'  {i:2d} = {name}')

print('\n' + '='*80)
print('  SAMPLE VALIDATION PREDICTIONS (25 examples)')
print('='*80)
print(f'{"#":<3} {"Text (truncated)":<42} {"Actual Issue":<25} {"Pred Issue":<25} {"Conf":>5} {"":>3}')
print('-' * 108)

for idx, rec in enumerate(sample_records[:25], 1):
    trunc = (rec['text'][:39] + '...') if len(rec['text']) > 42 else rec['text']
    match = 'OK' if rec['actual_issue'] == rec['pred_issue'] else 'X'
    print(f'{idx:<3d} {trunc:<42} {rec["actual_issue"]:<25} {rec["pred_issue"]:<25} {rec["confidence"]:>5.2f} {match:>3}')

print('\n[DIAGNOSTIC COMPLETE]')

### Step 9: ONNX Export & INT8 Quantization

Exports the fine-tuned checkpoint to ONNX (FP32, then INT8 dynamic quantization). The base model and output tag are derived from the *Config* cell (`FEEANA_MODEL_NAME`), so no flags are needed. The ONNX graph keeps the 2-input contract (`input_ids`, `attention_mask`) used by the browser runtime.

In [ ]:
!python scripts/training/export_model_onnx.py

### Step 10: Verify Exported Assets & Smoke Test

Lists the exported files (ONNX + tokenizer + config + label mappings) with sizes, then runs real-text inference through the INT8 model to validate output shapes and label decoding.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, "scripts/training")
from export_model_onnx import resolve_model_name, resolve_tag

EXPORTS = Path("scripts/training/exports")
model_name = resolve_model_name(None)
tag = resolve_tag(model_name, None)

tag_dir = EXPORTS / tag
if not tag_dir.exists():
    raise FileNotFoundError(f"{tag_dir} not found — did Step 9 run successfully?")

print(f"Model: {model_name} (tag: {tag})")
print(f"{"File":<45} {"Size":>10}")
print("-" * 58)
for f in sorted(tag_dir.iterdir()):
    if f.is_file():
        print(f"{f.name:<45} {f.stat().st_size / (1024 * 1024):>10.2f} MB")

int8_path = tag_dir / "int8.onnx"
if not int8_path.exists():
    raise FileNotFoundError(f"Expected INT8 model not found: {int8_path}")
print(f"\n[SUCCESS] INT8 model verified: {int8_path}")

In [ ]:
!python scripts/training/smoke_test_onnx.py